# Chain-of-Thought এবং Reasoning Prompts: Self-Consistency, কঠোরভাবে

Chain-of-Thought (Wei et al., 2022) এবং zero-shot CoT (Kojima et al., 2022, "Let's think step by step") হলো prompting-এর মাধ্যমেই মডেল থেকে একটা intermediate reasoning trace *উসকে বের করা*। Intermediate steps স্পষ্ট করে লেখা সত্যিই সাহায্য করে কি না — সেটা নির্দিষ্ট model ও task-এর ওপর নির্ভর করে এমনভাবে, যেটা একটা ছোট offline simulation দিয়ে সৎভাবে দেখানো যায় না (সেটা করতে হয় fake API call, নাহয় একটা LLM নতুন করে বানাতে হয়) — তাই এই script "steps লিখে দেখানো সাহায্য করে কি না" simulate করার চেষ্টাই করে না। বদলে সে CoT toolkit-এর সেই অংশটুকুর ওপর ফোকাস করে যেটা সুনির্দিষ্ট, প্রমাণযোগ্য আর সম্পূর্ণ self-contained একটা statistical দাবি: Wang et al.-এর (2022) SELF-CONSISTENCY পদ্ধতি, যা একই QUESTION-এর জন্য কয়েকটা independent reasoning path sample করে (যেমন temperature > 0 দিয়ে) আর একটা একক greedy decode-কে বিশ্বাস করার বদলে তাদের final answers-এর majority vote নেয়।

আমরা একটা reasoning sample-কে Bernoulli trial হিসেবে model করি: সেটা p probability দিয়ে সঠিক final answer-এ পৌঁছায়, অন্য প্রতিটি sample থেকে independent হয়ে (এটাই ঠিক সেই assumption যার ওপর self-consistency নির্ভর করে — sampled reasoning paths-গুলো এতোটাই independent হতে হবে যে তাদের errors সবগুলো একই দিকে না থাকে)। তারপর আমরা প্রমাণ করি — exact binomial formula দিয়েও, আবার এমন একটা Monte Carlo simulation দিয়েও যেটা ওর সাথে মিলতেই হবে — একটা Condorcet-Jury-Theorem-ধাঁচের ফলাফল:

- **p > 0.5** হলে, k টি independent sample-এর ওপর majority voting-এর accuracy একটা single sample-এর চেয়ে বেশি, এবং k বাড়ার সাথে সাথে strictly-increasing হয়ে 1.0-র দিকে যায়।
- **p < 0.5** হলে, majority voting জিনিসটা আরও খারাপ করে, k বাড়ার সাথে সাথে strictly decreasing হয়ে 0.0-র দিকে যায় — voting সেই bias-কেই amplify করে যেদিকে bias আগেই ঝুঁকে আছে, ভালো বা মন্দ যেভাবেই হোক।
- **p == 0.5** ঠিক হলে, voting কিছুই বদলায় না — amplify করার মতো কোনো signal নেই।

শেষে বাস্তব CoT task-এ self-consistency যেভাবে আচরণ করে তার একধাপ কাছাকাছি যাই: একটা একক "wrong" উত্তর-এর সাথে সঠিকটা প্রতিযোগিতা করার বদলে, ভুল reasoning paths-গুলো কয়েকটা ভিন্ন ভিন্ন wrong final answer-এ পৌঁছায় (একটা model সাধারণত ঠিক একই arithmetic ভুল দুবার করে না)। আমরা এমন একটা fragmented answer space-এর ওপর plurality voting simulate করি এবং দেখাই যে একই p-তে এটা binary case-কে ছাড়িয়ে যায় — রাজনৈতিক তাত্ত্বিকরা vote-splitting নামে যে ঘটনাটা জানেন, সেই একই স্বজ্ঞাগত কারণে: fragmented opposition-কে plurality-র পক্ষে হারানো unified-এর চেয়ে সহজ।

**Runtime:** CPU-তে কয়েক সেকেন্ড (pure Python + Monte Carlo, কোনো model training নেই)।

**চালানোর নিয়ম:**
- উপর থেকে নিচে cell-গুলো ক্রমান্বয়ে চালাও।
- আসল স্ক্রিপ্ট: `python example.py`

In [ ]:
import math
import random
from collections import Counter
random.seed(0)

## Section 1 — Binary majority voting-এর exact binomial formula

k টি independent Bernoulli(p) trial-এর majority সঠিক হওয়ার exact probability — ODD k-র জন্য (যাতে tie ভাঙতে না হয়)। এটাই ঠিক Condorcet Jury Theorem-এর সেই quantity।

In [ ]:
# ---------------------------------------------------------------------------
# 1. Binary majority voting-এর exact binomial formula
# ---------------------------------------------------------------------------

def majority_accuracy_exact(p, k):
    """k টি independent Bernoulli(p) trial-এর majority "correct" হওয়ার
    probability, ODD k-র জন্য (যাতে tie ভাঙতে না হয়)। এটাই ঠিক Condorcet
    Jury Theorem-এর পরিমাণ: k জন independent voter, প্রত্যেকে p probability
    দিয়ে সঠিক, majority rule।"""
    assert k % 2 == 1, "use odd k to avoid ties"
    half = k // 2
    total = 0.0
    for i in range(half + 1, k + 1):
        total += math.comb(k, i) * (p ** i) * ((1 - p) ** (k - i))
    return total

## Section 2 — একই quantity-র Monte Carlo simulation

Exact formula validate করার জন্য একটা independent simulation — step 5-এর কঠিন multi-way case-এ simulator-কে বিশ্বাস করার আগে নিশ্চিত হতে হবে যে simulator methodology নিজেই নির্ভুল।

In [ ]:
# ---------------------------------------------------------------------------
# 2. একই পরিমাণের Monte Carlo simulation, যেটা simulator methodology-কে
#    validate করে -- step 5-এর কঠিন multi-way case-এ বিশ্বাস করার আগে।
# ---------------------------------------------------------------------------

def simulate_binary_majority(p, k, num_trials):
    correct = 0
    for _ in range(num_trials):
        votes = [1 if random.random() < p else 0 for _ in range(k)]
        majority = 1 if sum(votes) > k / 2 else 0
        correct += majority
    return correct / num_trials

## Section 3 — Multi-way plurality voting

বাস্তব free-form answers-এ (সংখ্যা, expression) দুইটা ভিন্ন flawed reasoning path খুব কমই একই ভুল উত্তর-এ পৌঁছায়। তাই wrong mass `(1-p)`-কে একটা একক bucket-এর বদলে কয়েকটা distinct wrong label-এ ছড়িয়ে plurality vote নেওয়া হয় — free-form answers-এর ওপর বাস্তব self-consistency-র কাছাকাছি analogue।

In [ ]:
# ---------------------------------------------------------------------------
# 3. Multi-way plurality voting: wrong answers-গুলো একটা "other" bucket-এ
#    জড়ো না হয়ে কয়েকটা distinct wrong option-এ ছড়িয়ে থাকে --
#    free-form answers (সংখ্যা, expression) নিয়ে বাস্তব self-consistency-র
#    কাছাকাছি analogue, যেখানে দুইটা ভিন্ন ভুল reasoning path খুব কমই মিলে।
# ---------------------------------------------------------------------------

def simulate_plurality_vote(p, k, num_wrong_options, num_trials):
    """প্রতিটি sample p probability-তে "CORRECT", নাহলে num_wrong_options
    টা distinct wrong label-এর একটায় uniform random-ভাবে পড়ে। Final answer
    = k sample-এর মধ্যে সবচেয়ে বেশি vote পাওয়া label (ties uniform random-এ ভাঙা)।"""
    correct = 0
    wrong_labels = [f"wrong_{i}" for i in range(num_wrong_options)]
    for _ in range(num_trials):
        votes = []
        for _ in range(k):
            if random.random() < p:
                votes.append("CORRECT")
            else:
                votes.append(random.choice(wrong_labels))
        counts = Counter(votes)
        top_count = max(counts.values())
        winners = [label for label, c in counts.items() if c == top_count]
        winner = random.choice(winners)   # random tie-break
        correct += int(winner == "CORRECT")
    return correct / num_trials

## পুরো demonstration চালানো

নিচের cell-এ `main()` define করা আছে — পাঁচটা পরীক্ষা: exact formula বনাম simulation, helpful case, p=0.5 boundary, harmful case, আর vote-splitting। শেষ cell-এ `main()` কল হয়।

In [ ]:
def main():
    print("=" * 78)
    print("SELF-CONSISTENCY (Wang et al., 2022): MAJORITY VOTING OVER")
    print("INDEPENDENT REASONING SAMPLES, AS A CONDORCET-JURY-THEOREM RESULT")
    print("=" * 78)
    print("Model: each independently sampled reasoning path lands on the correct")
    print("final answer with probability p, and on a wrong answer otherwise.")
    print("Self-consistency draws k such samples and majority-votes the final")
    print("answer, instead of trusting a single greedy decode (k=1).")

    print("\n" + "=" * 78)
    print("1. EXACT FORMULA vs. MONTE CARLO SIMULATION (sanity check)")
    print("=" * 78)
    print("Before trusting simulation for the harder multi-way case below, we")
    print("confirm the Monte Carlo simulator agrees with the exact binomial")
    print("majority formula for the plain binary case.\n")
    NUM_TRIALS = 200_000
    print(f"{'p':>6}{'k':>6}{'exact':>12}{'simulated':>14}{'abs diff':>12}")
    for p_check, k_check in [(0.6, 5), (0.6, 15), (0.3, 5), (0.9, 7)]:
        exact = majority_accuracy_exact(p_check, k_check)
        sim = simulate_binary_majority(p_check, k_check, NUM_TRIALS)
        print(f"{p_check:>6.2f}{k_check:>6d}{exact:>12.4f}{sim:>14.4f}{abs(exact - sim):>12.4f}")
    print("\n-> Exact formula and simulation agree to within Monte Carlo noise")
    print("   (a few thousandths at 200,000 trials). The simulator is trustworthy.")

    print("\n" + "=" * 78)
    print("2. THE HELPFUL CASE: p > 0.5 -- voting drives accuracy UP towards 1.0")
    print("=" * 78)
    k_values = [1, 3, 5, 7, 15, 31, 51]
    good_ps = [0.55, 0.6, 0.7, 0.9]
    header = f"{'k':>6}  " + "".join(f"p={p:<10.2f}" for p in good_ps)
    print(header)
    up_trend_holds = True
    prev_row = None
    for k in k_values:
        row_vals = [majority_accuracy_exact(p, k) for p in good_ps]
        print(f"{k:>6}" + "".join(f"{v:>12.4f}" for v in row_vals))
        if prev_row is not None:
            for v_prev, v_now in zip(prev_row, row_vals):
                if v_now < v_prev - 1e-12:
                    up_trend_holds = False
        prev_row = row_vals
    print(f"\n-> For every p > 0.5 tested, accuracy at k=1 equals p itself, and rises")
    print(f"   MONOTONICALLY as k grows (confirmed: {up_trend_holds}), approaching 1.0.")
    print("   Even a weak reasoner (p=0.55, barely better than a coin flip per")
    print(f"   sample) reaches {majority_accuracy_exact(0.55, 51):.3f} accuracy at k=51 samples --")
    print("   this is the entire mechanism behind self-consistency's reported gains.")

    print("\n" + "=" * 78)
    print("3. THE BOUNDARY CASE: p == 0.5 exactly -- voting changes NOTHING")
    print("=" * 78)
    for k in [1, 5, 15, 51]:
        acc = majority_accuracy_exact(0.5, k)
        print(f"  k={k:<4d} accuracy = {acc:.4f}")
    print("\n-> At p=0.5 there is no signal to amplify -- each sample is a fair coin")
    print("   flip between right and wrong, and the majority of fair coin flips is")
    print("   itself just another fair coin flip. Accuracy stays pinned at 0.500")
    print("   for every k. Voting can only amplify a bias that already exists; it")
    print("   cannot manufacture one out of pure noise.")

    print("\n" + "=" * 78)
    print("4. THE HARMFUL CASE: p < 0.5 -- voting drives accuracy DOWN towards 0.0")
    print("=" * 78)
    bad_ps = [0.45, 0.4, 0.3, 0.1]
    header = f"{'k':>6}  " + "".join(f"p={p:<10.2f}" for p in bad_ps)
    print(header)
    down_trend_holds = True
    prev_row = None
    for k in k_values:
        row_vals = [majority_accuracy_exact(p, k) for p in bad_ps]
        print(f"{k:>6}" + "".join(f"{v:>12.4f}" for v in row_vals))
        if prev_row is not None:
            for v_prev, v_now in zip(prev_row, row_vals):
                if v_now > v_prev + 1e-12:
                    down_trend_holds = False
        prev_row = row_vals
    print(f"\n-> For every p < 0.5 tested, accuracy DECREASES monotonically as k grows")
    print(f"   (confirmed: {down_trend_holds}), approaching 0.0. This is the honest boundary")
    print("   condition of self-consistency: majority voting is not a free lunch --")
    print("   it amplifies whatever the per-sample accuracy already leans towards.")
    print("   If a reasoning strategy is WORSE than a coin flip per sample (a genuinely")
    print("   confusing or adversarial prompt can do this), sampling more and voting")
    print("   makes the final answer reliably WORSE, not better. Self-consistency is")
    print("   only worth applying when there is reason to believe p > 0.5 to begin")
    print("   with -- e.g. the base reasoning method already beats chance on the task.")

    print("\n" + "=" * 78)
    print("5. CLOSER TO REAL COT: WRONG ANSWERS SPLIT ACROSS MANY WRONG OPTIONS")
    print("=" * 78)
    print("Real free-form answers (a number, an expression) rarely repeat a wrong")
    print("value exactly -- different flawed reasoning paths tend to land on")
    print("DIFFERENT wrong answers, not one single competing wrong answer. We")
    print("simulate this by spreading the (1-p) wrong mass uniformly across")
    print("several distinct wrong labels, then take a PLURALITY vote.\n")
    p_demo = 0.4
    k_demo = 9
    NUM_TRIALS_PLURALITY = 100_000
    binary_acc = majority_accuracy_exact(p_demo, k_demo)
    print(f"p = {p_demo} (a MINORITY of samples are correct), k = {k_demo} samples")
    print(f"{'num_wrong_options':>20}{'plurality accuracy':>22}")
    print(f"{'1 (binary case)':>20}{binary_acc:>22.4f}   <- exact formula, for reference")
    plurality_accs = {}
    for num_wrong in [1, 2, 4, 8]:
        acc = simulate_plurality_vote(p_demo, k_demo, num_wrong, NUM_TRIALS_PLURALITY)
        plurality_accs[num_wrong] = acc
        print(f"{num_wrong:>20d}{acc:>22.4f}")

    fragmentation_helps = plurality_accs[8] > plurality_accs[1] + 0.02
    print(f"\n-> Even though p={p_demo} < 0.5 (a minority of samples are individually")
    print("   correct), spreading the wrong answers across more distinct options lets")
    print(f"   the single correct answer win the plurality more often: accuracy rises from")
    print(f"   {plurality_accs[1]:.3f} (1 wrong option, i.e. the binary case) to {plurality_accs[8]:.3f} (8 wrong")
    print(f"   options) at the same k and p (fragmentation helps: {fragmentation_helps}).")
    print("   This is vote-splitting: the correct answer is the single largest bloc")
    print("   because the WRONG mass is divided against itself. It is one honest")
    print("   reason self-consistency can outperform this script's own binary-case")
    print("   math on real tasks with open-ended (not binary) final answers --")
    print("   though note it does NOT rescue the p<0.5 case in general: with enough")
    print("   wrong options and low enough p, the wrong votes are still more numerous")
    print("   in total than the correct ones, just less concentrated on any one label.")

    print("\n" + "=" * 78)
    print("SUMMARY")
    print("=" * 78)
    print("Wei et al. (2022) show that PROMPTING a model to produce intermediate")
    print("reasoning steps (chain-of-thought) can raise per-sample accuracy p on")
    print("multi-step problems in the first place. Kojima et al. (2022) show a")
    print("surprisingly simple zero-shot version of this ('Let's think step by")
    print("step') recovers much of the benefit without any worked examples at all.")
    print("Wang et al. (2022) then layer SELF-CONSISTENCY on top: given any")
    print("reasoning method with per-sample accuracy p > 0.5, independently")
    print("resampling and majority-voting provably pushes accuracy higher still,")
    print("exactly as proven and measured above -- with the honest caveat that the")
    print("same mechanism backfires if p is not actually above 0.5 to begin with.")

In [ ]:
main()